In [8]:
import jsonlines
import pandas as pd
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    faithfulness
)
from datasets import Dataset
import matplotlib.pyplot as plt
import numpy as np


In [13]:
def load_jsonl_to_ragas_format(file_path):
    """Convert JSONL results to Ragas Dataset format"""
    data = {
        "question": [],
        "contexts": [],
        "answer": [],
        "ground_truth": []  # Optional, will be empty lists
    }
    
    with jsonlines.open(file_path, "r") as reader:
        for item in reader:
            data["question"].append(item["question"])
            data["contexts"].append(item["contexts"])  # Ragas expects list of lists
            data["answer"].append(item["answer"])
            data["ground_truth"].append("")  # Empty since we don't have ground truth
    
    return Dataset.from_dict(data)

In [14]:
def create_comparison_chart(df):
    """Create a bar chart comparing naive vs advanced RAG metrics"""
    metrics = df["Metric"].tolist()
    naive_scores = df["Naive RAG"].tolist()
    advanced_scores = df["Advanced RAG"].tolist()
    
    x = np.arange(len(metrics))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(10, 6))
    rects1 = ax.bar(x - width/2, naive_scores, width, label='Naive RAG')
    rects2 = ax.bar(x + width/2, advanced_scores, width, label='Advanced RAG')
    
    ax.set_ylabel('Score')
    ax.set_title('RAG Evaluation Metrics Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    
    # Add values on top of bars
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width()/2, height),
                        xytext=(0, 3),  # 3 points vertical offset
                        textcoords="offset points",
                        ha='center', va='bottom')
    
    autolabel(rects1)
    autolabel(rects2)
    
    fig.tight_layout()
    plt.savefig('rag_comparison.png')
    print("Comparison chart saved as rag_comparison.png")

In [15]:
def run_evaluation():
    """Run evaluation with Ragas"""
    # Load datasets
    print("Loading naive RAG results...")
    naive_dataset = load_jsonl_to_ragas_format("../results_naive.jsonl")
    
    print("Loading advanced RAG results...")
    advanced_dataset = load_jsonl_to_ragas_format("../results_advanced.jsonl")
    
    # Define metrics
    metrics = [answer_relevancy, context_precision, faithfulness]
    
    # Run evaluation
    print("Evaluating naive RAG...")
    naive_scores = evaluate(naive_dataset, metrics=metrics)
    
    print("Evaluating advanced RAG...")
    advanced_scores = evaluate(advanced_dataset, metrics=metrics)
    
    # Convert to DataFrame for easier comparison
    results = {
        "Metric": ["Answer Relevancy", "Context Precision", "Faithfulness"],
        "Naive RAG": [
            naive_scores["answer_relevancy"],
            naive_scores["context_precision"],
            naive_scores["faithfulness"]
        ],
        "Advanced RAG": [
            advanced_scores["answer_relevancy"],
            advanced_scores["context_precision"],
            advanced_scores["faithfulness"]
        ]
    }
    
    df = pd.DataFrame(results)
    print("\nEvaluation Results:")
    print(df)
    
    # Save to CSV
    df.to_csv("evaluation.csv", index=False)
    print("Results saved to evaluation.csv")
    
    # Create visualization
    create_comparison_chart(df)
    
    return naive_scores, advanced_scores

In [16]:
run_evaluation()

Loading naive RAG results...
Loading advanced RAG results...
Evaluating naive RAG...


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable